In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py

# In[1]:


import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request


import warnings
warnings.filterwarnings("ignore")
# In[2]:


# # creating engines and connections

# In[2]:




db = client.get_database('telematics_db')






# In[3]:


global count
count=0

global total_vehicle_for_analysis
total_vehicle_for_analysis=len(pd.read_sql("select vehicle_id from operator_vehicles where deboarding_time is null ",mysql_vds)  )       


# In[4]:


gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)


# In[5]:


def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r


# In[6]:


def loc_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]

    try:
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(vid)])
        #print("LOC PINGS A GAYI")
        
    except Exception as e:
        print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df


# In[7]:


def hb_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:        
        url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        hb_data=json.load(urllib.request.urlopen(url))
        hb=pd.DataFrame(hb_data['data'][str(vid)])
        #print("HB AAYA")

    except Exception as e:

        print("hb pings issue >> " + str(e))
        hb=pd.DataFrame()
        
    return hb


# In[8]:


def analysis_flow(l):
    lc=loc_pings(l)
    hb=hb_pings(l)
    
    #print("loc_pings length "+str(len(lc)))
    #print("hb_pings length "+str(len(hb)))
    global count
    count+=1
    vid=l[0]
    #print("- completion percentage : "+ str(round(100*count/total_vehicle_for_analysis,2))+"- number of vehicles : " + str(count))
    
    if (len(hb)>5) & (len(lc)>5):
        device_health=hb_lc(hb,lc,l)
        print(dt.datetime.now().strftime('%H:%M:%S:%s')+"- vehicle_id:"+str(vid)+" in analysis flow, went into hb_lc")
        return device_health
        #print("hb_lc main gaya")
        
    elif (len(lc)>5) & (len(hb)<5):
        device_health=lc_only(lc,l)
        print(dt.datetime.now().strftime('%H:%M:%S:%s')+"- vehicle_id:"+str(vid)+" in analysis flow, went into lc_only")
        return device_health
        #print("lc_only main gaya")

        
    elif (len(lc)<5) & (len(hb)>5):
        device_health=hb_only(hb,l)
        print(dt.datetime.now().strftime('%H:%M:%S:%s')+"- vehicle_id:"+str(vid)+" in analysis flow, went into hb_only")
        return device_health
    
    else:
        print(dt.datetime.now().strftime('%H:%M:%S:%s')+"- vehicle_id:"+str(vid)+" in analysis flow, no_data")
        
    #print("analysis_flow main toh gaya")
    #global count
    #count+=1
    #print("________{}NUMBER OF VEHICLES DONE_____".format(co
    


# In[9]:


def find_last_analysis(vid):
    data=pd.read_sql("select day from device_health where vehicle_id={}".format(vid),con=engine)
    try:    
        last_day=data.day.max()
        start=gettime(last_day)
    except:
        start=gettime(dt.date.today())-86400
        
    end=gettime(dt.date.today())
    l=[]
    threshhold=86400
    while(start+threshhold<end):
        l.append([vid,int(start),int(start+threshhold)])
        start+=threshhold
    l.append([vid,int(start),int(end)])
    
    return l


# In[10]:


def hb_only(hb,l):
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

        hb['created']=hb['created']/1000
        total_counts=0

        print('*** hb_only main hb pings= '+str(len(hb)))
        no_info_distance_travelled=None


        if('usedSat' in hb):
            sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]

        else:
            sat_used_values=[-1]*5



        if('gsmPct' in hb):
            gsm_pct_values=hb['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]

        else:
            gsm_pct_values=[-1]*5


        time_diff_cons_pings_values=[-1]*5

        server_delay_values=[-1]*6


        jump_values=[-1]*6

        driving_count=-1

        stop_count=-1


        driving_time=-1
        stop_time=-1

        day=getDay((day_start+day_end)/2)

        total_distace_travelled=-1


        hb=hb.sort_values('created',ascending=False)
        hb.reset_index(drop=True,inplace=True)
        hb['created_time_shift']=hb['created'].shift(1)
        hb.at[0,'created_time_shift']=hb.iloc[0]['created']
        hb['created_cons_time_diff']=hb['created_time_shift']-hb['created']
        created_cons_time_diff_values=hb[hb['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]

        live_time=hb[hb['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
        offline_time=hb[hb['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

        if (hb.created.min()-day_start)>=180:
            offline_time=offline_time+hb.created.min()-day_start
        else:
            live_time=live_time+hb.created.min()-day_start


        if (day_end-hb.created.max())>=180:
            offline_time=offline_time+day_end-hb.created.max()
        else :
            live_time=live_time+day_end-hb.created.max()

        no_info_duration=hb[hb['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

        if (hb.created.min()-day_start)>=3600:
            no_info_duration=no_info_duration+hb.created.min()-day_start

        elif (day_end-hb.created.max())>=3600:
            no_info_duration=no_info_duration+day_end-hb.created.max()


        jump_distance=-1
        jump_percent=-1

        expected_run_pings_percent=-1
        expected_stop_pings_percent=-1

        heart_beat_pings=len(hb)

        device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))



        no_info_instances=len(device_history)

        final_values=[vid,day,total_counts,heart_beat_pings]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]
        final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                        'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


        day_wise=pd.DataFrame([final_values],columns=final_column_names)
        print(day_wise.values())

        return day_wise
    except Exception as e:
        print(" ERROR IN HB ONLY ---------- "+str(e)+" for {}".format(l))
        
        return pd.DataFrame()


# In[11]:


#### FOLLOW #### this 

def lc_only(df,l):
    #if (1>0):
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

        print('*** lc_only main lc pings= '+str(len(df)))
        df.sort_values('time',ascending=False,inplace=True)

        df.reset_index(drop=True,inplace=True)

        try:
            no_info_epochs=[]

            device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))

            if len(device_history)>0:
                device_history.sort_values('fromTime',ascending=True,inplace=True)
                no_info_epochs=[]
                for b,brow in device_history.iterrows():
                    try:
                        
                        x=df.sort_values('time')
                        top_x=x[df['time']<brow['fromTime']].head(1)
                        no_info_start=top_x['time'].iloc[0]
                        
                        
                    except:
                        no_info_start=999999999999
                    try:
                        y=df.sort_values('time')
                        top_y=y[df['time']>brow['toTime']].tail(1)
                        no_info_end=top_y['time'].iloc[0]

                    except:
                        no_info_end=0

                    no_info_epochs.append([no_info_start,no_info_end])

            else:
                no_info_distance_travelled=None

        except Exception as e:
            device_history=pd.DataFrame()

            no_info_distance_travelled=None
            print('for the no_info_error for lc_only is ' + str(e))



        total_counts=len(df)



        if('usedSat' in df):
            sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            sat_used_values=[-1]*5



        if('gsmPct' in df):
            gsm_pct_values=df['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            gsm_pct_values=[-1]*5


        df.sort_values('time',ascending=False,inplace=True)

        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(1)

        df=df[1:]

        time_diff_cons=df['time1']-df['time']

        df.loc[:,'time_diff_cons_pings']=time_diff_cons

        time_diff_cons_pings_values=df.time_diff_cons_pings.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]



        df.loc[:,'server_created_time']=df['created']/1000 #.apply(getTime)



        df.loc[:,'server_delay']=df['server_created_time']-df['time']

        server_delay_values=df.server_delay.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


        loc_list=df[['location','loc2']].values.tolist()

        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)

        df.loc[:,'distance']=dis

        no_info_distance_travelled=0
        for epoch in no_info_epochs:
            no_info_distance_travelled+=df[df['time']>=epoch[0]][df['time']<=epoch[1]]['distance'].sum()

        jump_values=df[df['distance']>500]['distance'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]


        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)

        driving_count=len(df[df['moving_status']=='running'])

        stop_count=len(df[df['moving_status']=='stop'])


        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

        day=getDay((df['time'].min()+df['time'].max())/2)

        total_distace_travelled=df['distance'].sum()


        df=df.sort_values('server_created_time',ascending=False)

        df.reset_index(drop=True,inplace=True)


        df['created_time_shift']=df['server_created_time'].shift(1)
        df.at[0,'created_time_shift']=df.iloc[0]['server_created_time']


        df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']


        live_time=df[df['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
        offline_time=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

        created_cons_time_diff_values=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]



        if (df.time.min()-day_start)>=180:
            offline_time=offline_time+df.time.min()-day_start
        else:
            live_time=live_time+df.time.min()-day_start


        if (day_end-df.time.max())>=180:
            offline_time=offline_time+day_end-df.time.max()
        else :
            live_time=live_time+day_end-df.time.max()


        jump_distance=df[df['distance']>500]['distance'].sum()
        jump_percent=100*jump_distance/total_distace_travelled

        expected_run_pings_percent=1000*driving_count/driving_time
        expected_stop_pings_percent=3000*stop_count/stop_time


        no_info_duration=df[df['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

        if (df.time.min()-day_start)>=3600:
            no_info_duration=no_info_duration+df.time.min()-day_start

        if (day_end-df.time.max())>=3600:
            no_info_duration=no_info_duration+day_end-df.time.max()

        no_info_instances=len(device_history)




        final_values=[vid,day,total_counts,None]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]


        final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                 'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


        day_wise=pd.DataFrame([final_values],columns=final_column_names)


        return day_wise
    #else:
    except Exception as e:
        print("LC ONLY MAIN ISSUE AAYA HAI------------" +str(e)+" for {}".format(l) )
        day_wise=pd.DataFrame()
        return day_wise


# In[12]:


def hb_lc(hb,df,l):
    try:

        vid=l[0]
        day_start=l[1]
        day_end=l[2]    

        hb['created']=hb['created']/1000
        total_counts=len(df)

        print('***hb_lc main loc pings= '+str(len(df)))
        df=df.sort_values('time',ascending=False)
        df.reset_index(drop=True,inplace=True)

        try:
            no_info_epochs=[]
            device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))

            if len(device_history)>0:
                device_history.sort_values('fromTime',ascending=True,inplace=True)
                for b,brow in device_history.iterrows():
                    try:
                        x=df.sort_values('time')
                        top_x=x[df['time']<brow['fromTime']].head(1)
                        no_info_start=top_x['time'].iloc[0]


#                       no_info_start=df.sort_values('time')[df['time']<brow['fromTime']].head(1)['time'].iloc[0]
                    except:
                        no_info_start=999999999999
                    try:
                        y=df.sort_values('time')
                        top_y=y[df['time']>brow['toTime']].tail(1)
                        no_info_end=top_y['time'].iloc[0]

#                       no_info_end=df.sort_values('time')[df['time']>brow['toTime']].tail(1)['time'].iloc[0]

                    except:
                        no_info_end=0

                    no_info_epochs.append([no_info_start,no_info_end])

            else:
                no_info_distance_travelled=None

        except Exception as e:
            device_history=pd.DataFrame()
            no_info_distance_travelled=None
            print('for the no_info_error for hb_lc is ' + str(e))



        if('usedSat' in df):
            sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        elif('usedSat' in hb):
            sat_used_values=hb['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            sat_used_values=[-1]*5



        if('gsmPct' in hb):
            gsm_pct_values=hb['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        elif ('gsmPct' in df):
            gsm_pct_values=df['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
        else:
            gsm_pct_values=[-1]*5


        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(1)

        df=df[1:]

        time_diff_cons=df['time1']-df['time']

        df.loc[:,'time_diff_cons_pings']=time_diff_cons

        time_diff_cons_pings_values=df.time_diff_cons_pings.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


        df.loc[:,'server_created_time']=df['created']/1000 #.apply(getTime)



        df.loc[:,'server_delay']=df['server_created_time']-df['time']

        server_delay_values=df.server_delay.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]


        loc_list=df[['location','loc2']].values.tolist()

        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)

        df['distance']=dis

        no_info_distance_travelled=0
        for epoch in no_info_epochs:
            no_info_distance_travelled+=df[df['time']>=epoch[0]][df['time']<=epoch[1]]['distance'].sum()


        jump_values=df[df['distance']>500]['distance'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]


        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)

        driving_count=len(df[df['moving_status']=='running'])

        stop_count=len(df[df['moving_status']=='stop'])


        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()

        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()

        day=getDay((df['time'].min()+df['time'].max())/2)

        total_distace_travelled=df['distance'].sum()


        df=df.sort_values('server_created_time',ascending=False)

        df.reset_index(drop=True,inplace=True)
        df['created_time_shift']=df['server_created_time'].shift(1)
        df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']

        hb=hb.sort_values('created',ascending=False)
        hb.reset_index(drop=True,inplace=True)
        hb['created_time_shift']=hb['created'].shift(1)
        hb.at[0,'created_time_shift']=hb.iloc[0]['created']
        hb['created_cons_time_diff']=hb['created_time_shift']-hb['created']

        live_time=hb[hb['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
        offline_time=hb[hb['created_cons_time_diff']>180]['created_cons_time_diff'].sum()

        if (hb.created.min()-day_start)>=180:
            offline_time=offline_time+hb.created.min()-day_start
        else:
            live_time=live_time+hb.created.min()-day_start


        if (day_end-hb.created.max())>=180:
            offline_time=offline_time+day_end-hb.created.max()
        else :
            live_time=live_time+day_end-hb.created.max()

        no_info_duration=hb[hb['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()

        if (df.time.min()-day_start)>=3600:
            no_info_duration=no_info_duration+df.time.min()-day_start

        elif (day_end-df.time.max())>=3600:
            no_info_duration=no_info_duration+day_end-df.time.max()


        df['created_time_shift']=df['server_created_time'].shift(1)
        df.at[0,'created_time_shift']=df.iloc[0]['server_created_time']
        df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']
        created_cons_time_diff_values=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]
        jump_distance=df[df['distance']>500]['distance'].sum()
        jump_percent=100*jump_distance/total_distace_travelled

        expected_run_pings_percent=1000*driving_count/driving_time
        expected_stop_pings_percent=3000*stop_count/stop_time

        heart_beat_pings=len(hb)
        no_info_instances=len(device_history)
        final_values=[vid,day,total_counts,heart_beat_pings]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration,no_info_instances]
        final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']


        day_wise=pd.DataFrame([final_values],columns=final_column_names)
        
    

        return day_wise
    except Exception as e:
        print("HB_LC MAIN ISSUE AAYA HAI---------" + str(e)+" for  {}".format(l))
        day_wise=pd.DataFrame()
        return day_wise



# In[ ]:


















# In[13]:


#engine.execute(" delete from device_health where day<{}".format(str("'"+str((datetime.now()-timedelta(days=30)).strftime("%Y-%m-%d"))+str("'"))))


# In[14]:



vehicle_count=pd.read_sql("select max(vehicle_id) from operator_vehicles",con=mysql_vds)
vehicle_count=vehicle_count['max(vehicle_id)'].iloc[0]



for k in range(vehicle_count,0,-1000):
    if 1>0:
    #try:
        
        aggregate=pd.read_sql("select * from operator_vehicles where deboarding_time is null and vehicle_id>{} and vehicle_id<{} ;".format(k-1000,k),mysql_vds).rename(index=str,columns={"vehicle_id":"vehicleId"})

        start=gettime(dt.date.today())-19800-86400
        end=gettime(dt.date.today())-19800
        vehicle_list=aggregate.vehicleId.tolist()        
        
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(50)
        reuslts=pool.map(find_last_analysis,vehicle_list)
        pool.close()
        pool.join()

        vid_start_end=[]
        for i in reuslts:
            for k in i:
                vid_start_end.append(k)
        
        
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(10)
        results2=pool.map(analysis_flow,vid_start_end)
        pool.close()
        pool.join()

        data=pd.concat(results2)
        

        db = client.get_database('telematics_db')
        
                                                                                                                     

                                                                                                                      


        data=data.round(0)

        vehicle_details=pd.read_sql("select vehicle_id,boarding_time,vehicle_number,data_source,operator_id,new_device,installation_subtype,device_id,is_force_live,operator_name,operator_phone_number,sim_id,model_name,sales_dept,sim_number,sim_phone_number,imei from vehicle_details",con=engine)

        data=pd.merge(data,vehicle_details,on='vehicle_id',how='left')

        devices=pd.read_sql("select created as device_created_time,imei,new_device as device_table_new_device,version as device_version,purchase_date as device_purchase_date from devices",con=mysql_vds)

        data=pd.merge(data,devices, on='imei',how='left')
        
        print(dt.datetime.now().strftime('%H:%M:%S:%s')+"  " +str(k)+ "range vehicles have total data for:"+str(len(data))+" vehicles to be appended")

        data.set_index(['vehicle_number','day','vehicle_id']).to_sql("device_health",con=engine,if_exists='append',chunksize=200)
        print("***************** 1000 batch done **************** {}".format(k))

    #except Exception as e:
    else:
        print("KOI ISSUE AAYA HAI, ONE RANGE OF 1000 ( range {} to next 1000) VEHICLES NAHI HUE HONGE, the error being >>".format(k)+ str(e))
        
        



In [ ]:


import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request


import warnings
warnings.filterwarnings("ignore")
# In[2]:


# # creating engines and connections

# In[2]:




db = client.get_database('telematics_db')





gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)



In [ ]:
def no_data(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    day=getDay((day_start+day_end)/2)


    final_values=[vid,day,0,0]+[-1]*46
    final_column_names=['vehicle_id','day','total_raw_pings','heart_beat_pings',
                                        'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']

    day_wise=pd.DataFrame([final_values],columns=final_column_names)
    return day_wise


In [ ]:
        from multiprocessing.dummy import Pool as ThreadPool
        pool = ThreadPool(10)
        results2=pool.map(no_data,vid_start_end)
        pool.close()
        pool.join()


In [ ]:
def

In [ ]:
'vehicle_id','day','total_raw_pings','heart_beat_pings',
                                        'usedSat_mean', 'usedSat_1_pctl', 'usedSat_5_pctl', 'usedSat_10_pctl', 'usedSat_50_pctl',
                                       'Gsm_mean', 'Gsm_1_pctl', 'Gsm_5_pctl', 'Gsm_10_pctl', 'Gsm_50_pctl',
                                       'cnsctv_time_diff_mean', 'cnsctv_time_diff_50_pctl', 'cnsctv_time_diff_80_pctl', 'cnsctv_time_diff_95_pctl', 'cnsctv_time_diff_99_pctl',
                                       'server_delay_mean', 'server_delay_50_pctl', 'server_delay_80_pctl', 'server_delay_95_pctl', 'server_delay_99_pctl',
                                       'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50_pctl', 'jump_500mts_80_pctl', 'jump_500mts_95_pctl', 'jump_500mts_99_pctl',
                                       'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                                       'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50_pctl', 'created_cnsctv_80_pctl', 'created_cnsctv_95_pctl', 'created_cnsctv_99_pctl',
                                       'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration','no_info_instances']
